# BGG API

Exploring the BGG XML API2 for requests

| Date | User | Change Type | Remarks |  
| ---- | ---- | ----------- | ------- |
| 23/03/2026   | Martin | Create  |  | 

# Content

* [Request & XML](#request--xml)

# Request & XML

Pulling and exploring the returned XML data

In [12]:
import os
import requests
import pandas as pd
from xml.etree import ElementTree
from dotenv import load_dotenv

load_dotenv()

True

1. Get the list of games that can be pulled 

In [21]:
df_bg = pd.read_csv("../data/boardgames_ranks.csv")
df_bg.head()

,id,name,yearpublished,rank,bayesaverage,average,usersrated,is_expansion,abstracts_rank,cgs_rank,childrensgames_rank,familygames_rank,partygames_rank,strategygames_rank,thematic_rank,wargames_rank,pulled
0,224517,Brass: Birmingham,2018,1,8.39530,8.56746,57227,0,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0
1,342942,Ark Nova,2021,2,8.35524,8.54188,59656,0,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,0
2,161936,Pandemic Legacy: Season 1,2015,3,8.34784,8.50432,57122,0,NaN,NaN,NaN,NaN,NaN,3.0,1.0,NaN,0
3,174430,Gloomhaven,2017,4,8.30257,8.54350,66778,0,NaN,NaN,NaN,NaN,NaN,5.0,2.0,NaN,0
4,316554,Dune: Imperium,2020,5,8.22374,8.41359,57460,0,NaN,NaN,NaN,NaN,NaN,7.0,NaN,NaN,0


In [ ]:
def get_bgg_id(df: pd.DataFrame) -> int:
  """
  Get the highest ranking ID of the board game that hasn't been pulled yet

  Args:
      df (pd.DataFrame): boardgames_ranks df

  Returns:
      int: BGG ID
  """
  filtered_df = df[df['pulled'] != 1]
  return filtered_df['id'].to_numpy()[0]

In [ ]:
def update_bgg_id(df:pd.DataFrame, id: int) -> pd.DataFrame:
  """
  Update the pulled column of the dataframe based on the BGG ID

  Args:
      df (pd.DataFrame): boardgames_ranks df
      id (int): BGG ID

  Returns:
      pd.DataFrame: Updated dataframe based on BGG ID
  """
  df.loc[df['id'] == id, 'pulled'] = 1
  return df

2. Pulling and compiling data

In [71]:
def get_suggested_players(tree):
  poll = tree.find(".//poll[@name='suggested_numplayers']")
  suggested = max(
    poll.findall("results"),
    key=lambda r: int(r.find("result[@value='Best']").get("numvotes"))
  )
  return suggested.get("numplayers")

def get_suggested_age(tree):
  poll = tree.find(".//poll[@name='suggested_playerage']")
  suggested = max(
    poll.findall(".//result"),
    key=lambda r: int(r.get("numvotes"))
  )
  return suggested.get("value")

def get_link_type_list(tree, tag):
  v = [l.get('value') for l in tree.findall(f".//link[@type='{tag}']")]
  i = [l.get('id') for l in tree.findall(f".//link[@type='{tag}']")]
  return i, v
  


In [ ]:
# tree.findall(".//link[@type='boardgamecategory']")

[str, str, str, str, str, str]

In [74]:
BGG_ID = 224517
TOKEN = os.environ['BGG_TOKEN']
PAGE = 1
URL = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&stats=1&comments=1&page={PAGE}"

response = requests.get(
  URL,
  headers={"Authorization": f"Bearer {TOKEN}"}
)

if response.status_code == 200:
  tree = ElementTree.fromstring(response.content)
else:
  print("Failed to retrieve data")

# BGG Info
bgg_info = {}
bgg_info['id'] = BGG_ID
bgg_info['name'] = tree.find(".//name[@type='primary']").get('value')
bgg_info['description'] = tree.find(".//description").text.strip('\n')
bgg_info['year_published'] = tree.find('.//yearpublished').get('value')
## Players
bgg_info['min_players'] = int(tree.find(".//minplayers").get('value'))
bgg_info['max_players'] = int(tree.find(".//maxplayers").get('value'))
bgg_info['suggested_players'] = get_suggested_players(tree)
## Playing time
bgg_info['playing_time'] = int(tree.find(".//playingtime").get('value'))
bgg_info['min_playing_time'] = int(tree.find(".//minplaytime").get('value'))
bgg_info['max_playing_time'] = int(tree.find(".//maxplaytime").get('value'))
## Age
bgg_info['min_age'] = int(tree.find(".//minage").get('value'))
bgg_info['suggested_age'] = get_suggested_age(tree)
## Tags
bgg_info['category_ids'], bgg_info['categories'] = get_link_type_list(tree, 'boardgamecategory')
bgg_info['mechanic_ids'], bgg_info['mechanics'] = get_link_type_list(tree, 'boardgamemechanic')
bgg_info['family_ids'], bgg_info['families'] = get_link_type_list(tree, 'boardgamefamily')
bgg_info['designer_ids'], bgg_info['designers'] = get_link_type_list(tree, 'boardgamedesigner')
bgg_info['artist_ids'], bgg_info['artists'] = get_link_type_list(tree, 'boardgameartist')
## Ratings
bgg_info['num_ratings'] = int(tree.find(".//usersrated").get('value'))
bgg_info['rating'] = float(tree.find(".//average").get('value'))
bgg_info['bayes_rating'] = float(tree.find(".//bayesaverage").get('value'))
bgg_info['complexity'] = float(tree.find(".//averageweight").get('value'))

In [75]:
bgg_info

{'id': 224517,
 'name': 'Brass: Birmingham',
 'description': "Brass: Birmingham is an economic strategy game sequel to Martin Wallace's 2007 masterpiece, Brass. Brass: Birmingham tells the story of competing entrepreneurs in Birmingham during the industrial revolution between the years of 1770 and 1870.\n\nIt offers a very different story arc and experience from its predecessor. As in its predecessor, you must develop, build and establish your industries and network in an effort to exploit low or high market demands. The game is played over two halves: the canal era (years 1770-1830) and the rail era (years 1830-1870). To win the game, score the most VPs. VPs are counted at the end of each half for the canals, rails and established (flipped) industry tiles.\n\nEach round, players take turns according to the turn order track, receiving two actions to perform any of the following actions (found in the original game):\n\n1) Build - Pay required resources and place an industry tile.\n2) Ne

In [ ]:
# Comments
MAX_PAGES = 10
MIN_WORDS = 20
URL = f"https://boardgamegeek.com/xmlapi2/thing?id={BGG_ID}&type=boardgame&stats=1&comments=1&page={PAGE}"

comments = []
all_comments = tree.findall(".//comment")

for comment in all_comments:
  text = comment.get('value').strip()
  if len(text.split(' ')) < MIN_WORDS:
    continue

  info = {
    'rating': comment.get('rating'),
    'comment': comment.get('value')
  }
  comments.append(info)

In [80]:
comments

[{'rating': 'N/A',
  'comment': 'Kickstarter Project http://kck.st/2oFYhmV - Backed: 04/17/2017 - Funded: 05/11/2017 - Pledge Level: Pledge $132 CAD or more: "CA$ 132 or more  About $99  ~ Brass ~ Full Bundle ($100 USD) The Complete Brass Series includes both Brass:Lancashire and Brass:Birmingham levels above. This bundle also saves you money on shipping!  *Shipping charged after campaign ends. See Shipping section for details." - Payment: $132 CAD - Survey: 08/09/2017 backerkit - Projected Delivery: 01/2018 - Shipped:   Status: DELIVERED 08/07/2018'},
 {'rating': 'N/A',
  'comment': 'Ich habe: - Basisspiel "Birmingham" (Englische Version mit deutschem Regelheft zum Ausdrucken. Habe es 1x ausgedruckt für Lancashire & Birmingham zusammen). [Wohn-W-Offen] - Icon-Clay-Auffüllpaket auf 100 Münzen für Birmingham (je 1x für Lancashire & Birmingham) (Addon). [Wohn-N-Schrank8] - Playmat "Birmingham" (Erhalten von der Spieleschmiede) (Addon).  Ich habe auch: - Basisspiel "Lancashire" (Englische

In [ ]:

for page in range(1, MAX_PAGES):
  response = requests.get(
    URL,
    headers={"Authorization": f"Bearer {TOKEN}"}
  )

  if response.status_code == 200:
    tree = ElementTree.fromstring(response.content)
  else:
    print("Failed to retrieve data")

In [2]:
%watermark

Last updated: 2025-06-18T19:03:45.452311+08:00

Python implementation: CPython
Python version       : 3.11.9
IPython version      : 8.31.0

Compiler    : MSC v.1938 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 20
Architecture: 64bit

